# Wykład 4 — Kiedy program podejmuje decyzję?

## W4.0 — Od wartości do wyboru ściezki wykonania

W W2 obliczaliśmy wartości i poznaliśmy `bool`; w W3 odczytywaliśmy dane i korzystaliśmy z prostego `if` jako zabezpieczenia.
Przejdziemy dalej i zapytamy: **która instrukcja zostanie wykonana, a której program w ogóle nie odwiedzi?**

**Cel:** przewidzieć wartość warunku, rzeczywistą kolejność ewaluacji i wybraną drogę przez `if`/`else`/`?:`/`switch`.

### Pierwsze doświadczenie: ta sama liczba, różny przebieg

Przewidź, co zostanie wypisane dla `x = -3`, a potem dla `x = 3`. W tym doświadczeniu używamy już znanego z W3 prostego `if`, bez wyjaśniania całej składni po raz drugi.

In [ ]:
#include <stdio.h>

int main(void)
{
    int x = -3;
    printf("Przed warunkiem\n");
    if (x < 0) {
        printf("x jest ujemne\n");
    }
    printf("Po warunku\n");
    return 0;
}

### Co zaobserwowaliśmy?

Warunek może zmienić **zbiór wykonanych instrukcji**. Niezależnie od wybranej drogi program kontynuuje działanie za blokiem `if`. Zamień `x = -3` na `x = 3` i przewidź wynik przed ponownym uruchomieniem.

**Powrót po spirali:** W2.9 zawiera wyniki porównań, W2.11 priorytety, a W3.5–W3.7 wykorzystuje `if` do kontroli wejścia.

## W4.1 — Co właściwie znaczy „warunek”?

Z W2.9: porównania dają `int` równy 0 albo 1. W warunkach C **zero oznacza fałsz**, a **każda wartość niezerowa — prawdę**.

Wyrażenie oblicza wartość; instrukcja `if` wykorzystuje ją do wyboru ścieżki.

## W4.2 — Łączymy warunki

**Problem:** czy `x` należy do przedziału `[0, 10]`?

W C zapisujemy: `(x >= 0) && (x <= 10)`.

**Przewidź** wyniki dla `x = -1, 0, 7, 10, 11`; zmieniaj jedną wartość.

**Pułapka:** `0 <= x <= 10` nie sprawdza przedziału. W C oznacza `(0 <= x) <= 10`.

In [ ]:
#include <stdio.h>

int main(void)
{
    int x = 7;
    int w_przedziale = (x >= 0) && (x <= 10);
    printf("x=%d, w_przedziale=%d\n", x, w_przedziale);
    return 0;
}

### `&&`, `||`, `!` — tabela prawdy

| p | q | \&\& | \|\| | !p |
|:-:|:-:|:--:|:--:|:--:|
| 0 | 0 | 0 | 0 | 1 |
| 0 | 1 | 0 | 1 | 1 |
| 1 | 0 | 0 | 1 | 0 |
| 1 | 1 | 1 | 1 | 0 |

`&&` wymaga obu warunków; `||` co najmniej jednego; `!` neguje. Wyniki tych operatorów mają typ `int` i wynoszą 0 albo 1. **Każdy niezerowy operand to prawda** — także ujemny.

**Sprawdź w C:** dlaczego `-2 && 7` daje `1`, a nie `7`?

In [ ]:
#include <stdio.h>

void wiersz(int p, int q)
{
    printf("%d %d  %d   %d   %d\n", p, q, p && q, p || q, !p);
}

int main(void)
{
    puts("p q AND OR NOT_p");
    wiersz(0, 0);
    wiersz(0, 1);
    wiersz(1, 0);
    wiersz(1, 1);
    printf("-2 && 7 = %d, -2 || 0 = %d, !(-2) = %d\n", -2 && 7, -2 || 0, !(-2));
    return 0;
}

### Negacja warunku — prawa de Morgana

`!(p && q)` jest równoważne `(!p || !q)`.  
`!(p || q)` jest równoważne `(!p && !q)`.

Warunek `!(x >= 0 && x <= 10)` jest równoważny `(x < 0) || (x > 10)`.

**Sprawdź** dla `x = -1, 0, 10, 11`.

In [ ]:
#include <stdio.h>

int main(void)
{
    int x = 11;
    int w = (x >= 0) && (x <= 10);
    int poza1 = !w;
    int poza2 = (x < 0) || (x > 10);
    printf("x=%d, w=%d, !w=%d, poza=%d\n", x, w, poza1, poza2);
    return 0;
}

## W4.3 — Priorytet i kolejność obliczania

Wśród operatorów logicznych priorytet to `!` → `&&` → `||`.

`p || q && r` oznacza `p || (q && r)`, a nie `(p || q) && r`.

**Przewidź** wyniki dla `p = 1, q = 0, r = 0`.

In [ ]:
#include <stdio.h>

int main(void)
{
    int p = 1, q = 0, r = 0;
    printf("p || (q && r) = %d\n", p || (q && r));
    printf("(p || q) && r = %d\n", (p || q) && r);
    return 0;
}

### Short-circuit: prawa strona nie zawsze jest obliczana

Dla `p && q`: gdy `p` jest fałszem, `q` zostaje pominięte.  
Dla `p || q`: gdy `p` jest prawdą, `q` zostaje pominięte.

`&&` i `||` obliczają lewy operand przed prawym. **Priorytet operatorów to nie kolejność ewaluacji.**

Funkcja `prawy_operand()` wypisuje komunikat, jeżeli zostanie wywołana. **Przewidź, ile razy.**

In [ ]:
#include <stdio.h>

int prawy_operand(int wartosc)
{
    puts("Obliczam prawy operand");
    return wartosc;
}

int main(void)
{
    int p = 0;
    int wynik = p && prawy_operand(1);
    printf("0 && P: %d\n", wynik);

    p = 1;
    wynik = p && prawy_operand(1);
    printf("1 && P: %d\n", wynik);

    p = 1;
    wynik = p || prawy_operand(0);
    printf("1 || P: %d\n", wynik);

    p = 0;
    wynik = p || prawy_operand(0);
    printf("0 || P: %d\n", wynik);
    return 0;
}

### Zastosowanie: unikamy dzielenia przez zero

`(mianownik != 0) && (licznik / mianownik > 2)` nie wykonuje dzielenia, gdy mianownik jest zerem.

**Sprawdź** dla `mianownik = 0, 2, 5`. Zastąpienie `&&` przez `&` nie jest bezpieczne: `&` nie ma short-circuit.

In [ ]:
#include <stdio.h>

int main(void)
{
    int licznik = 10, mianownik = 0;
    int wynik = (mianownik != 0) && (licznik / mianownik > 2);
    printf("mianownik=%d, wynik=%d\n", mianownik, wynik);
    return 0;
}

### Pytania kontrolne

1. Oblicz `!(3 < 5 || 4 > 8)`. Czy oba porównania zostaną wykonane?
2. Czy dla `p = 1` wyrażenie `p || prawy_operand(0)` wywoła funkcję?
3. Zapisz warunek: `x` jest poza przedziałem `[-2, 3)`. Sprawdź wartości brzegowe.

**Dalej:** `if` i `else` wykorzystają wyniki tych wyrażeń do wyboru gałęzi.

## W4.4 — Instrukcje warunkowe

Wyrażenie logiczne ma wartość, a `if` wybiera instrukcje do wykonania.

**Klamry `{ }` tworzą blok.** Bez nich `if` steruje tylko następną instrukcją. Przewidź wynik dla `a = -1`, potem `1` i `0`.

In [ ]:
#include <stdio.h>
int main(void)
{
    int a = -1;
    if (a > 0)
        puts("dodatnie");
    if (a < 0) {
        puts("ujemne");
        puts("druga instrukcja bloku");
    }
    puts("koniec");
    return 0;
}

### Niezależne `if` a łańcuch `else if`

Dla `a == b` w pierwszej części wykonają się **trzy** bloki, a w drugiej — tylko **jeden**.

**Przewidź** wydruk; następnie ustaw `a = 7` i `a = 3`.

In [ ]:
#include <stdio.h>
int main(void)
{
    int a = 5, b = 5;
    puts("Niezalezne if:");
    if (a == b) puts("rowne");
    if (a >= b) puts("a >= b");
    if (a <= b) puts("a <= b");

    puts("Lancuch else if:");
    if (a == b) puts("rowne");
    else if (a > b) puts("a > b");
    else puts("a < b");
    return 0;
}

### Zagnieżdżone `if`: do którego należy `else`?

`else` wiąże się z najbliższym niepowiązanym `if`. Klamry pokazują strukturę jednoznacznie.

**Przewidź** wynik dla `(a,b) = (2,0), (2,1), (-2,0)`.

In [ ]:
#include <stdio.h>
int main(void)
{
    int a = 2, b = 0;
    if (a > 0) {
        if (b > 0) {
            puts("a > 0 i b > 0");
        } else {
            puts("a > 0 i b <= 0");
        }
    } else {
        puts("a <= 0");
    }
    return 0;
}

### Pułapka: `=` nie oznacza `==`

`if (a = 7)` **przypisuje** wartość `7` do `a`. Wynik przypisania jest niezerowy, więc warunek jest prawdziwy.

Zapis jest składniowo poprawny, ale GCC ostrzega (`-Wparentheses`): to celowy eksperyment. **Przewidź** końcową wartość `a`.

In [ ]:
#include <stdio.h>
int main(void)
{
    int a = 9;
    if (a = 7) {
        puts("warunek prawdziwy");
    }
    printf("a=%d\n", a);
    return 0;
}

## W4.5 — Operator warunkowy `?:`

`warunek ? wartosc_gdy_prawda : wartosc_gdy_falsz` jest **wyrażeniem**. Obliczana jest tylko wybrana z dwóch gałęzi.

**Porównaj** wynik z `if/else`. Zamień `b = 5` na `9`, potem `7`.

In [ ]:
#include <stdio.h>
int main(void)
{
    int a = 10, b = 5, c = 7;
    int d_if;
    if (b > c) {
        d_if = a + b;
    } else {
        d_if = a + c;
    }
    int d_op = a + (b > c ? b : c);
    printf("if=%d, operator=%d\n", d_if, d_op);
    return 0;
}

## W4.6 — `switch`: wybór według wartości

`switch` wybiera miejsce rozpoczęcia wykonania na podstawie **wyrażenia całkowitego**. Etykiety `case` muszą być stałymi wyrażeniami całkowitymi; `default` obsługuje pozostałe wartości.

**Przewidź** wydruk dla `kod = 1, 2, 3, 8`. `break` kończy `switch`.

In [ ]:
#include <stdio.h>
int main(void)
{
    int kod = 2;
    switch (kod) {
        case 1: puts("jeden"); break;
        case 2: puts("dwa"); break;
        case 3: puts("trzy"); break;
        default: puts("inny kod");
    }
    return 0;
}

### Co się stanie bez `break`?

Po wejściu do `case 2` wykonanie przechodzi też przez `case 3`. To **fall-through**, a nie ponowny test warunku. Komentarz `fall through` oznacza celowe przejście dla GCC.

**Przewidź** wynik dla `kod = 2`, potem `1`, `3` i `8`.

In [ ]:
#include <stdio.h>
int main(void)
{
    int kod = 2;
    switch (kod) {
        case 1: puts("jeden"); break;
        case 2: puts("dwa"); /* fall through */
        case 3: puts("trzy"); break;
        default: puts("inny kod");
    }
    return 0;
}

### Pytania kontrolne

1. Czy trzy niezależne `if` mogą wykonać trzy bloki?
2. Czy wcięcia decydują, z którym `if` łączy się `else`?
3. Dlaczego `if (a = 7)` zmienia `a`?
4. Co wypisze `switch` dla `kod = 2` bez `break`?

**Dalej:** grafy przepływu sterowania (checkpoint W4.4).

## W4.7 — Studium przypadku: równanie kwadratowe

Wracamy do wyróżnika z W3.7: `delta = b*b - 4*a*c`. **Nowość nie polega na wzorze**, tylko na decyzji, które obliczenia wolno wykonać.

**Do W4.5:** czytelne drzewo decyzji dla niepoprawnego wejścia; `a == 0`; wariantów liniowych; `delta < 0`, `delta == 0`, `delta > 0`. Testy na liczbach dokładnie reprezentowanych, później odsyłacz do W2.6. Prosty wariant w notebooku użyje ustalonych danych/`sscanf`, a program z klawiaturą pozostanie oddzielnym eksperymentem terminalowym.

## W4.8 — Co umiemy? Co dalej?

**Pytania kontrolne (do uzupełnienia przy W4.2–W4.7):**

1. Czy z wartości całego warunku można wywnioskować, które operandy obliczono?
2. Ile gałęzi może wykonać pojedynczy łańcuch `if/else if/else`?
3. Czym różni się `?:` od `if`?
4. Co zmienia usunięcie `break` z `switch`?

**Dalej:** W5 — powtarzanie instrukcji za pomocą pętli. Analiza ścieżek wykonania przyda się również tam.

**Uzupełnienie W4.6 (checkpoint):** przekazywanie argumentów przez wartość, zasięg i przesłanianie nazw, zmienne globalne; w C **nie ma przeciążania funkcji** przez różne listy parametrów. Nie rozszerzamy teraz głównego wykładu o ponowny kurs funkcji.